# Datathon 2026 — Career Success Score Tahmini

**Yaklaşım özeti:**
1. Tabular özellik mühendisliği (beceri agregatları, rol-beceri eşleşmesi, etkileşimler)
2. `mentor_feedback_text` için TF-IDF (kelime+karakter) Ridge OOF tahmini + çok dilli sentence-transformer embedding Ridge OOF tahmini + SVD/PCA bileşenleri
3. LightGBM / XGBoost / CatBoost (5-fold, 2'şer seed) + MLP topluluğu
4. OOF üzerinde Nelder-Mead ile optimize edilmiş harman ağırlıkları, tahminler [0,100] aralığına kırpılır

5-fold CV MSE: **~76.15**

## Özellik mühendisliği

In [ ]:
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

SKILLS = ['coding_score','problem_solving_score','data_structures_score','sql_score',
          'machine_learning_score','backend_score','frontend_score','cloud_score','devops_score']

ROLE_SKILLS = {
 'Backend Developer': ['backend_score','sql_score','data_structures_score','coding_score'],
 'Frontend Developer': ['frontend_score','coding_score','problem_solving_score'],
 'Software Developer': ['coding_score','data_structures_score','problem_solving_score','backend_score','frontend_score'],
 'Data Scientist': ['machine_learning_score','sql_score','problem_solving_score'],
 'Data Analyst': ['sql_score','machine_learning_score','problem_solving_score'],
 'AI Engineer': ['machine_learning_score','coding_score','data_structures_score'],
 'Cloud Engineer': ['cloud_score','devops_score','backend_score'],
 'DevOps Engineer': ['devops_score','cloud_score','backend_score'],
 'Machine Learning Engineer': ['machine_learning_score','coding_score','data_structures_score'],
 'Full Stack Developer': ['backend_score','frontend_score','coding_score','sql_score'],
 'Mobile Developer': ['coding_score','frontend_score','problem_solving_score'],
}

def engineer(df):
    df = df.copy()
    df['skill_mean'] = df[SKILLS].mean(axis=1)
    df['skill_max'] = df[SKILLS].max(axis=1)
    df['skill_min'] = df[SKILLS].min(axis=1)
    df['skill_std'] = df[SKILLS].std(axis=1)
    # role-matched skills
    rm = np.zeros(len(df)); rmax = np.zeros(len(df))
    for role, cols in ROLE_SKILLS.items():
        m = (df['target_role']==role).values
        if m.sum():
            rm[m] = df.loc[m, cols].mean(axis=1)
            rmax[m] = df.loc[m, cols].max(axis=1)
    df['role_skill_mean'] = rm
    df['role_skill_max'] = rmax
    df['role_skill_gap'] = df['role_skill_mean'] - df['skill_mean']
    # interview / soft
    df['interview_mean'] = df[['technical_interview_score','hr_interview_score']].mean(axis=1)
    df['soft_mean'] = df[['communication_score','teamwork_score','leadership_score','presentation_score']].mean(axis=1)
    # experience
    df['total_projects'] = df['real_client_project_count'] + df['freelance_project_count']
    df['exp_score'] = (df['real_client_project_count']*2 + df['freelance_project_count']
                       + df['internship_count'] + df['hackathon_awards']*2)
    df['internship_total'] = df['internship_count'] * df['internship_duration_months'].fillna(0)
    df['github_activity'] = df['github_repo_count'] * (1+df['github_avg_stars'].fillna(0))
    df['interview_ratio'] = df['interviews_attended'] / (df['applications_sent']+1)
    df['years_since_grad'] = df['application_year'] - df['graduation_year']
    # key interactions
    df['pq_x_ti'] = df['project_quality_score'] * df['technical_interview_score']
    df['pq_x_skill'] = df['project_quality_score'] * df['skill_mean']
    df['pq_x_role'] = df['project_quality_score'] * df['role_skill_mean']
    df['ti_x_skill'] = df['technical_interview_score'] * df['skill_mean']
    df['pq_x_comm'] = df['project_quality_score'] * df['communication_score']
    df['portfolio_x_github'] = df['portfolio_score'].fillna(0) * np.log1p(df['github_repo_count'])
    df['n_missing'] = df[['english_exam_score','internship_duration_months','portfolio_score',
                          'github_avg_stars','open_source_contribution_count',
                          'linkedin_profile_score','hr_interview_score']].isna().sum(axis=1)
    df['text_len'] = df['mentor_feedback_text'].str.len()
    df['text_words'] = df['mentor_feedback_text'].str.split().str.len()
    return df

def text_features(tr_txt, te_txt, y, n_svd=64, seed=42):
    """Returns (svd_tr, svd_te, oof_pred, te_pred) from TF-IDF."""
    tv = TfidfVectorizer(max_features=60000, ngram_range=(1,3), sublinear_tf=True, min_df=2)
    A = tv.fit_transform(pd.concat([tr_txt, te_txt]))
    Atr, Ate = A[:len(tr_txt)], A[len(tr_txt):]
    svd = TruncatedSVD(n_components=n_svd, random_state=seed)
    S = svd.fit_transform(A)
    Str, Ste = S[:len(tr_txt)], S[len(tr_txt):]
    # OOF ridge on tfidf
    kf = KFold(5, shuffle=True, random_state=seed)
    oof = np.zeros(len(tr_txt)); te_pred = np.zeros(Ate.shape[0])
    for tr_i, va_i in kf.split(Atr):
        r = Ridge(alpha=1.0)
        r.fit(Atr[tr_i], y[tr_i])
        oof[va_i] = r.predict(Atr[va_i])
        te_pred += r.predict(Ate)/5
    return Str, Ste, oof, te_pred


## Pipeline: metin özellikleri, modeller, harman, submission

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from scipy.optimize import minimize
from scipy.sparse import hstack
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold


SEED = 42
N_FOLDS = 5
CAT_COLS = ['department', 'university_tier', 'target_role', 'hobby',
            'preferred_social_media_platform']
DROP = ['student_id', 'career_success_score', 'mentor_feedback_text']

tr = pd.read_csv('data/train.csv', encoding='utf-8-sig')
te = pd.read_csv('data/test.csv', encoding='utf-8-sig')
y = tr['career_success_score'].values
kf = KFold(N_FOLDS, shuffle=True, random_state=SEED)
folds = list(kf.split(tr))

# ---------- text features ----------
txt_tr = tr['mentor_feedback_text'].fillna('')
txt_te = te['mentor_feedback_text'].fillna('')
all_txt = pd.concat([txt_tr, txt_te])

tw = TfidfVectorizer(max_features=80000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
tc = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=80000,
                     sublinear_tf=True, min_df=2)
A = hstack([tw.fit_transform(all_txt), tc.fit_transform(all_txt)]).tocsr()
Atr, Ate = A[:len(tr)], A[len(tr):]

tfidf_oof = np.zeros(len(tr)); tfidf_te = np.zeros(len(te))
for tr_i, va_i in folds:
    r = Ridge(alpha=2.0)
    r.fit(Atr[tr_i], y[tr_i])
    tfidf_oof[va_i] = r.predict(Atr[va_i])
    tfidf_te += r.predict(Ate) / N_FOLDS
print('tfidf ridge MSE:', mean_squared_error(y, tfidf_oof))

import os
if os.path.exists('cache_emb_tr.npy'):
    E_tr = np.load('cache_emb_tr.npy')
    E_te = np.load('cache_emb_te.npy')
else:
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    E_tr = st.encode(txt_tr.tolist(), batch_size=128)
    E_te = st.encode(txt_te.tolist(), batch_size=128)
    np.save('cache_emb_tr.npy', E_tr)
    np.save('cache_emb_te.npy', E_te)
emb_oof = np.zeros(len(tr)); emb_te = np.zeros(len(te))
for tr_i, va_i in folds:
    r = Ridge(alpha=10.0)
    r.fit(E_tr[tr_i], y[tr_i])
    emb_oof[va_i] = r.predict(E_tr[va_i])
    emb_te += r.predict(E_te) / N_FOLDS
print('emb ridge MSE:', mean_squared_error(y, emb_oof))

svd = TruncatedSVD(n_components=64, random_state=SEED)
S = svd.fit_transform(A)
Str, Ste = S[:len(tr)], S[len(tr):]

pca = PCA(n_components=32, random_state=SEED)
P_all = pca.fit_transform(np.vstack([E_tr, E_te]))
Ptr, Pte = P_all[:len(tr)], P_all[len(tr):]

# ---------- tabular matrix ----------
def build(df, S_, P_, tf_pred, em_pred):
    f = engineer(df).drop(columns=[c for c in DROP if c in df.columns]).reset_index(drop=True)
    f = pd.concat([f,
                   pd.DataFrame(S_, columns=[f'svd_{i}' for i in range(S_.shape[1])]),
                   pd.DataFrame(P_, columns=[f'emb_{i}' for i in range(P_.shape[1])])], axis=1)
    f['text_pred_tfidf'] = tf_pred
    f['text_pred_emb'] = em_pred
    return f

X = build(tr, Str, Ptr, tfidf_oof, emb_oof)
Xte = build(te, Ste, Pte, tfidf_te, emb_te)

# ---------- models ----------
def run_lgb(seed):
    oof = np.zeros(len(X)); tep = np.zeros(len(Xte))
    Xl, Xtl = X.copy(), Xte.copy()
    for c in CAT_COLS:
        Xl[c] = Xl[c].astype('category')
        Xtl[c] = Xtl[c].astype('category').cat.set_categories(Xl[c].cat.categories)
    for tr_i, va_i in folds:
        m = lgb.LGBMRegressor(n_estimators=6000, learning_rate=0.02, num_leaves=63,
                              colsample_bytree=0.7, subsample=0.8, subsample_freq=1,
                              min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
                              random_state=seed, verbose=-1)
        m.fit(Xl.iloc[tr_i], y[tr_i], eval_set=[(Xl.iloc[va_i], y[va_i])],
              callbacks=[lgb.early_stopping(300, verbose=False)])
        oof[va_i] = m.predict(Xl.iloc[va_i])
        tep += m.predict(Xtl) / N_FOLDS
    return oof, tep

def run_xgb(seed):
    oof = np.zeros(len(X)); tep = np.zeros(len(Xte))
    Xx, Xtx = X.copy(), Xte.copy()
    for c in CAT_COLS:
        Xx[c] = Xx[c].astype('category')
        Xtx[c] = Xtx[c].astype('category').cat.set_categories(Xx[c].cat.categories)
    for tr_i, va_i in folds:
        m = xgb.XGBRegressor(n_estimators=6000, learning_rate=0.02, max_depth=6,
                             colsample_bytree=0.7, subsample=0.8, min_child_weight=5,
                             reg_alpha=0.1, reg_lambda=1.0, enable_categorical=True,
                             tree_method='hist', early_stopping_rounds=300,
                             random_state=seed)
        m.fit(Xx.iloc[tr_i], y[tr_i], eval_set=[(Xx.iloc[va_i], y[va_i])], verbose=False)
        oof[va_i] = m.predict(Xx.iloc[va_i])
        tep += m.predict(Xtx) / N_FOLDS
    return oof, tep

def run_cat(seed):
    oof = np.zeros(len(X)); tep = np.zeros(len(Xte))
    Xc, Xtc = X.copy(), Xte.copy()
    for c in CAT_COLS:
        Xc[c] = Xc[c].astype(str)
        Xtc[c] = Xtc[c].astype(str)
    for tr_i, va_i in folds:
        m = CatBoostRegressor(iterations=10000, learning_rate=0.03, depth=6,
                              l2_leaf_reg=3, cat_features=CAT_COLS,
                              early_stopping_rounds=400, random_seed=seed, verbose=False)
        m.fit(Xc.iloc[tr_i], y[tr_i], eval_set=(Xc.iloc[va_i], y[va_i]))
        oof[va_i] = m.predict(Xc.iloc[va_i])
        tep += m.predict(Xtc) / N_FOLDS
    return oof, tep

def run_mlp(seed):
    from sklearn.neural_network import MLPRegressor
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.impute import SimpleImputer
    num_tr = X.drop(columns=CAT_COLS)
    num_te = Xte.drop(columns=CAT_COLS)
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    C_tr = ohe.fit_transform(X[CAT_COLS]); C_te = ohe.transform(Xte[CAT_COLS])
    imp = SimpleImputer(strategy='median')
    N_tr = imp.fit_transform(num_tr); N_te = imp.transform(num_te)
    sc = StandardScaler()
    Mtr = sc.fit_transform(np.hstack([N_tr, C_tr]))
    Mte = sc.transform(np.hstack([N_te, C_te]))
    oof = np.zeros(len(X)); tep = np.zeros(len(Xte))
    for tr_i, va_i in folds:
        m = MLPRegressor(hidden_layer_sizes=(256, 128), alpha=1e-3,
                         learning_rate_init=1e-3, batch_size=256, max_iter=200,
                         early_stopping=True, n_iter_no_change=15, random_state=seed)
        m.fit(Mtr[tr_i], y[tr_i])
        oof[va_i] = m.predict(Mtr[va_i])
        tep += m.predict(Mte) / N_FOLDS
    return oof, tep

oofs, teps, names = [], [], []
for seed in (42, 2026):
    for name, fn in (('lgb', run_lgb), ('xgb', run_xgb), ('cat', run_cat)):
        o, t = fn(seed)
        oofs.append(o); teps.append(t); names.append(f'{name}_{seed}')
        print(f'{name}_{seed} MSE: {mean_squared_error(y, np.clip(o, 0, 100)):.4f}')
o, t = run_mlp(42)
oofs.append(o); teps.append(t); names.append('mlp_42')
print(f'mlp_42 MSE: {mean_squared_error(y, np.clip(o, 0, 100)):.4f}')

O = np.vstack(oofs).T
T = np.vstack(teps).T
np.save('cache_O.npy', O); np.save('cache_T.npy', T)

def loss(w):
    w = np.abs(w); w = w / w.sum()
    return mean_squared_error(y, np.clip(O @ w, 0, 100))

best = None
inits = [np.ones(O.shape[1])] + [np.random.RandomState(s).rand(O.shape[1]) + 0.5
                                 for s in range(3)]
for init in inits:
    res = minimize(loss, init, method='Nelder-Mead',
                   options={'maxiter': 8000, 'xatol': 1e-6, 'fatol': 1e-9})
    if best is None or res.fun < best.fun:
        best = res
w = np.abs(best.x); w = w / w.sum()
print('weights:', dict(zip(names, np.round(w, 4))))
print('BLEND CV MSE:', loss(best.x))

pred = np.clip(T @ w, 0, 100)
sub = pd.DataFrame({'student_id': te['student_id'], 'career_success_score': pred})
sub.to_csv('submission.csv', index=False)
np.save('cache_blend_oof.npy', np.clip(O @ w, 0, 100))
print('submission.csv written', sub.shape)
